The data that the model will process will be of a structured or semi-structured nature, such as transaction records or company web logs that will be in CSV or JSON format. The data we will use for preprocessing will be from Kaggle, in particular: https://www.kaggle.com/datasets/shriyashjagtap/fraudulent-e-commerce-transactions as of 29/06/2026 19:00. The train_transaction.csv dataset is used.

In [1]:
!pip install pandas numpy matplotlib seaborn
!pip install category_encoders

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm

In [3]:

train_trans_path = 'train_transaction.csv'
data = pd.read_csv(train_trans_path)

In [4]:
import pandas as pd

#we obtain the total number of nulls per column
null_counts = data.isnull().sum()

#we calculate the fraud rate specifically where there are nulls
#we filter only columns that have at least one null to avoid division by zero
cols_with_nulls = null_counts[null_counts > 0].index

#for each column, we calculate the mean of 'isFraud' only in the rows where that column is null
fraud_rate_in_nulls = data[cols_with_nulls].isnull().apply(lambda x: data.loc[x, 'isFraud'].mean())

#we create a summary DataFrame to see it all together
summary_df = pd.DataFrame({
    'Null Proportion': data[cols_with_nulls].isnull().mean(),
    'Fraud in Nulls Rate': fraud_rate_in_nulls
}).sort_values(by='Fraud in Nulls Rate', ascending=False)

#we add the global rate for reference
print(f"Global Fraud Rate: {data['isFraud'].mean():.4f}\n")
display(summary_df)

Global Fraud Rate: 0.0350



,Null Proportion,Fraud in Nulls Rate
V317,0.000020,0.166667
V316,0.000020,0.166667
V312,0.000020,0.166667
V309,0.000020,0.166667
V310,0.000020,0.166667
...,...,...
V185,0.763235,0.021284
V169,0.763235,0.021284
V170,0.763235,0.021284
R_emaildomain,0.767516,0.020819


In [5]:
overall_fraud_rate = data['isFraud'].mean()

#we define the limits of the criterion to remove columns
lower_bound = 0.5 * overall_fraud_rate
upper_bound = 1.5 * overall_fraud_rate

#we identify the columns that meet both conditions
cols_to_drop_candidate = summary_df[
    (summary_df['Null Proportion'] > 0.25) &
    (summary_df['Fraud in Nulls Rate'] > lower_bound) &
    (summary_df['Fraud in Nulls Rate'] < upper_bound)
].index.tolist()

#we filter to keep only those that currently exist in 'data'
cols_to_drop = [c for c in cols_to_drop_candidate if c in data.columns]

print(f"Overall fraud rate: {overall_fraud_rate:.4f}")
print(f"Exclusion range: [{lower_bound:.4f} - {upper_bound:.4f}]")
print(f"Number of variables to remove identified: {len(cols_to_drop)}")

if len(cols_to_drop) > 0:
    #we safely remove the columns
    data_drop = data.drop(columns=cols_to_drop)
    print(f"Variables dropped successfully.")
else:
    print("No variables found to drop (perhaps they were already dropped).")

print(f"\nNew dataset size: {data_drop.shape}")

Tasa de fraude global: 0.0350
Rango de exclusión: [0.0175 - 0.0525]
Número de variables a eliminar identificadas: 208
Variables eliminadas correctamente.

Nuevo tamaño del dataset: (590540, 186)


In [6]:
#we identify categorical variables that are still in the filtered dataset
cat_cols = data_drop.select_dtypes(include=['object', 'category', 'str']).columns

#we filter only those that have null values
cat_with_nulls = [col for col in cat_cols if data_drop[col].isnull().any()]

#we calculate the number of unique categories for each one
cat_summary = []
for col in cat_with_nulls:
    n_unique = data_drop[col].nunique()  # nunique por defecto no cuenta nulos
    cat_summary.append({
        'Variable': col,
        'Unique Categories': n_unique,
        'Example Values': data_drop[col].dropna().unique()[:5].tolist()
    })

#we show the result
cat_summary_df = pd.DataFrame(cat_summary).sort_values(by='Unique Categories', ascending=False)
print(f"Found {len(cat_with_nulls)} categorical variables with nulls.")
display(cat_summary_df)

Se encontraron 7 variables categóricas con nulos.


,Variable,Unique Categories,Example Values
2,P_emaildomain,59,"[gmail.com, outlook.com, yahoo.com, mail.com, ..."
0,card4,4,"[discover, mastercard, visa, american express]"
1,card6,4,"[credit, debit, debit or credit, charge card]"
3,M1,2,"[T, F]"
4,M2,2,"[T, F]"
5,M3,2,"[T, F]"
6,M6,2,"[T, F]"


In [7]:
def make_hour_feature(df, tname='TransactionDT'):
    hours = df[tname] / 3600
    encoded_hours = np.floor(hours) % 24
    return encoded_hours

#defragment the DataFrame before adding new columns
data_drop = data_drop.copy()
data_drop = data_drop.assign(hour_of_the_day=make_hour_feature(data))

In [8]:
#apply cyclical encoding to the time of day
data_drop = data_drop.assign(
    hour_sin=np.sin(2 * np.pi * data_drop['hour_of_the_day'] / 24),
    hour_cos=np.cos(2 * np.pi * data_drop['hour_of_the_day'] / 24)
)

In [9]:
#drop the original time variables
cols_to_remove = ['TransactionDT', 'hour_of_the_day']
data_drop = data_drop.drop(columns=cols_to_remove)

print(f"Removed variables: {cols_to_remove}")
print(f"New number of columns: {data_drop.shape[1]}")

Variables eliminadas: ['TransactionDT', 'hour_of_the_day']
Nuevo número de columnas: 187


In [10]:
#identify categorical variables with exactly 2 unique values
cat_cols_current = data_drop.select_dtypes(include=['object', 'category', 'str']).columns
binary_cats = [col for col in cat_cols_current if data_drop[col].nunique() == 2]

#apply the encoding
for col in binary_cats:
    #get the two unique categories (excluding nulls)
    unique_vals = data_drop[col].dropna().unique()
    mapping = {unique_vals[0]: 0, unique_vals[1]: 1}

    #apply the mapping and fill nulls with -1
    data_drop[col] = data_drop[col].map(mapping).fillna(-1).astype(int)
    print(f"Variable '{col}' encoded: {mapping} (nulls -> -1)")

#verify the result
display(data_drop[binary_cats].head())

Variable 'M1' codificada: {'T': 0, 'F': 1} (nulos -> -1)
Variable 'M2' codificada: {'T': 0, 'F': 1} (nulos -> -1)
Variable 'M3' codificada: {'T': 0, 'F': 1} (nulos -> -1)
Variable 'M6' codificada: {'T': 0, 'F': 1} (nulos -> -1)


,M1,M2,M3,M6
0,0,0,0,0
1,-1,-1,-1,0
2,0,0,0,1
3,-1,-1,-1,1
4,-1,-1,-1,-1


In [11]:
from category_encoders import TargetEncoder

#define the desired columns and filter those that actually exist in data_drop
target_cols_potential = ['ProductCD', 'card4', 'card6', 'P_emaildomain']
target_cols = [col for col in target_cols_potential if col in data_drop.columns]

if len(target_cols) > 0:
    #initialize the encoder with smoothing
    encoder = TargetEncoder(cols=target_cols, smoothing=10.0, handle_missing='value')

    #fit and transform
    data_drop[target_cols] = encoder.fit_transform(data_drop[target_cols], data_drop['isFraud'])

    print(f"Target Encoding with smoothing applied to: {target_cols}")
    display(data_drop[target_cols].head())
else:
    print("Error: The specified columns were not found in 'data_drop'.")
    print("Available columns:", data_drop.columns.tolist())

Target Encoding con suavizado aplicado a: ['ProductCD', 'card4', 'card6', 'P_emaildomain']


,ProductCD,card4,card6,P_emaildomain
0,0.020399,0.077282,0.066785,0.029538
1,0.020399,0.034331,0.066785,0.043542
2,0.020399,0.034756,0.024263,0.094584
3,0.020399,0.034331,0.024263,0.022757
4,0.047662,0.034331,0.066785,0.043542


In [12]:
#final check of data types
remaining_objects = data_drop.select_dtypes(include=['object']).columns

print(f"Remaining 'object' type columns: {len(remaining_objects)}")
if len(remaining_objects) > 0:
    print("Variables to review:", remaining_objects.tolist())
else:
    print("Perfect! There are no text variables left in the dataset.")

display(data_drop.info(max_cols=10))

Columnas tipo 'object' restantes: 0
¡Perfecto! Ya no quedan variables de texto en el dataset.
<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 187 entries, TransactionID to hour_cos
dtypes: float64(180), int64(7)
memory usage: 842.5 MB


None

In [13]:
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#separate TransactionID and isFraud so they aren't imputed or used in PCA
transaction_id = data_drop['TransactionID']
is_fraud = data_drop['isFraud']

#select only numerical columns excluding TransactionID and isFraud
numerical_cols = data_drop.select_dtypes(include=np.number).columns.tolist()
numerical_cols.remove('TransactionID')
numerical_cols.remove('isFraud')

df_numerical = data_drop[numerical_cols].copy()

print(f"Número de columnas numéricas para imputar y transformar: {len(numerical_cols)}")
print("Columnas excluidas: 'TransactionID', 'isFraud'")
print(f"Nulos totales antes de imputar: {df_numerical.isnull().sum().sum()}")

Número de columnas numéricas para imputar y transformar: 185
Columnas excluidas: 'TransactionID', 'isFraud'
Nulos totales antes de imputar: 5569489


In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#prepare data in float32
X = df_numerical.astype(np.float32)

#identify columns with null values
cols_with_nulls = X.columns[X.isnull().any()].tolist()

#create indicator variables for each column with nulls using pd.concat
null_indicators = pd.DataFrame(
    {f'{col}_is_null': X[col].isnull().astype(int) for col in cols_with_nulls},
    index=X.index
)
X = pd.concat([X, null_indicators], axis=1)

#impute by median
imputer_median = SimpleImputer(strategy='median')
X_imputed = imputer_median.fit_transform(X)
df_imputed = pd.DataFrame(X_imputed, columns=X.columns, index=data_drop.index)

#standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_imputed)

#PCA retaining 95% of explained variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_cols = [f'PC{i+1}' for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_cols, index=data_drop.index)

#reconstruct final dataset excluding TransactionID and isFraud from the process
data_processed_pca = pd.concat([
    transaction_id,
    is_fraud,
    df_pca
], axis=1)

print("Median imputation applied.")
print(f"Indicator variables created for: {cols_with_nulls}")
print(f"Remaining nulls after imputation: {df_imputed.isnull().sum().sum()}")
print(f"PCA applied. Retained components: {df_pca.shape[1]}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.sum():.4f}")
display(data_processed_pca.head())
print(f"New dataset size: {data_processed_pca.shape}")

Imputación por mediana aplicada.
Variables indicatriz creadas para: ['card2', 'card3', 'card5', 'addr1', 'addr2', 'D1', 'D10', 'D15', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V279', 'V280', 'V281', 'V282', 'V283', 'V284', 'V285', 'V286', 'V287', 

,TransactionID,isFraud,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC49,PC50,PC51,PC52,PC53,PC54,PC55,PC56,PC57,PC58
0,2987000,0,-2.812304,-0.208356,-1.988135,0.769578,1.005296,-0.050541,-0.396390,-0.170980,...,-2.760648,3.971832,1.342395,0.244731,0.869270,-2.970130,1.850355,2.902999,-0.156138,-0.005063
1,2987001,0,-2.848866,-0.186640,-0.767854,0.292770,-0.087629,0.051613,-0.454461,-0.636806,...,0.536992,-2.101837,-1.492194,-0.471217,0.077704,0.645065,-0.435126,0.449426,-0.153716,-0.014690
2,2987002,0,-2.952438,-0.203841,-1.942251,0.754457,0.813366,-0.057893,-0.353220,-0.181182,...,0.545377,-0.939924,-0.365373,-0.377931,0.059966,-0.083328,-0.250236,-0.209644,-0.048682,-0.021864
3,2987003,0,-2.826651,-0.242448,-0.054284,-0.044544,3.426651,-0.274201,-0.135622,0.405952,...,-0.592504,-0.469711,-1.153049,-0.288519,-0.183219,-0.815181,-0.194468,-0.933502,0.359622,-0.295053
4,2987004,0,21.221497,-0.030112,-0.689567,0.200455,-0.846914,-0.124807,-0.551059,-0.208578,...,0.420572,-2.033961,-1.242360,-0.440225,0.052740,-0.038309,-0.141688,-0.173991,0.013975,-0.031114


Nuevo tamaño del dataset: (590540, 60)


## 1. Case statement and justification of the approach

The chosen scenario is that of an e-commerce company that processes large volumes of digital transactions and faces a recurring problem of financial fraud. Each operation generates structured and semi-structured data: identifiers, amounts, payment methods, email domains, temporal variables, and multiple technical attributes, some specific to the platform and others from third-party systems. Although in the preprocessing we will only treat tabular data.

From a business perspective, fraud implies direct losses through chargebacks, additional costs for manual review, reputational risk, and deterioration of the experience of legitimate customers. Therefore, the problem is formulated not only as a binary classification task, but as a need to design a system that detects anomalous operations on time, minimizes false negatives, and integrates into an infrastructure capable of growing and remaining operational under load.

The project requires Big Data because traditional systems collapse given the magnitude and nature of the data, which are numerous and in real-time. Specifically, millions of daily transactions are processed, exceeding the capacity of conventional servers; we need detection to be in real-time (thus we will need streaming processing) for the sake of the legitimate user and the company; data comes from heterogeneous sources such as CSV sales files, JSON user logs, and geolocation data; there are noises and inconsistencies that must be processed in real-time and on a large scale; not to forget the value and competitive advantage of implementing a Big Data solution to the problem.

The system we propose has different objectives:
* Scalability: design an operational architecture capable of scaling horizontally as sales increase without redesigning the system.
* Availability and resilience: guarantee fault tolerance that allows continuous 24/7 operation.
* Reliability and improvement of the detection system: reduce the false positive rate through automated preprocessing. We establish the F1-score and MCC metrics to measure the quality of the system.
* Automation: we will implement an end-to-end pipeline covering the entire process from data ingestion to the issuance and visualization of alerts for the security team.



## 2. Data Science process applied to fraud

### 2.1. Methodological framework and lifecycle

To structure the project, CRISP-DM is adopted as a reference framework, given that it organizes the work into six phases: business understanding, data understanding, preparation, modeling, evaluation, and deployment. This approach is particularly suitable when every decision must be clearly justified and traceability must be maintained in regulated or high-impact contexts, such as financial fraud detection. The case shows that the process is not linear: decisions made when preparing the data can be reviewed after exploration, and the modeling results can suggest going back to refine the selection of variables or even reformulate the problem. This cyclical vision is consistent with a real data science process, where it is emphasized that the decisive factor is an iterative and well-articulated flow between acquisition, cleaning, analysis, and communication.

### 2.2. Data acquisition and initial understanding

The dataset used comes from Kaggle and contains transactions with an `isFraud` label, which allows working on a supervised binary classification problem with imbalanced classes. The data is presented in a tabular format, with hundreds of numerical variables (many of them derived, such as the `V` ones), categorical attributes (payment methods, email domains, M indicators) and a temporal variable `TransactionDT` that indexes the transaction's position in time. The initial understanding reveals several critical features: a high proportion of null values in numerous columns, variable cardinality in categories, strong imbalance between fraudulent and legitimate transactions, and high dimensionality. These elements connect with the description of typical difficulties in data processing: noise, missing information, inconsistent formats, and structural complexity, all of them present in real data science cases.

### 2.3. Preprocessing: decisions and fundamentals

In the technical notebook we have already implemented an extensive preprocessing as a coherent sequence of quality treatment and complexity reduction. In a first phase, the number of nulls per column is quantified and the fraud rate is calculated specifically in the rows with missing values, comparing it with the overall fraud rate. This comparison allows identifying columns in which nulls do not add a relevant differentiating signal, but do introduce noise and processing cost. With this criterion, 208 variables with more than 25% nulls and a fraud rate in missing records close to the overall rate are eliminated, reducing the dataset to 186 columns. From the point of view of data processing theory, this decision represents a dimensionality reduction at the attribute level, which seeks to preserve the essential and dispense with the superfluous, reorganizing the space to maintain relevant informative content based on firm and explicit criteria.

In the next phase, temporal feature engineering is applied. The `TransactionDT` variable is transformed into an hour of the day and cyclically encoded using sine and cosine functions, respecting the circular nature of the 24 hours and avoiding artificial jumps between consecutive values such as 23:00 and 00:00. This way, underlying relationships are more easily capturable by the models.

For categorical variables, two complementary strategies are adopted. Binary variables M1, M2, M3, M6 are encoded through direct mapping to integers, handling nulls with a special value, -1, while attributes such as `ProductCD`, `card4`, `card6` and `Pemaildomain` are transformed using target encoding with smoothing, leveraging their relationship with the `isFraud` label without causing the dimensionality explosion typical of one-hot schemes. This use of supervised encoding is consistent with the idea of leveraging information from the target variable in categorical attributes of moderate cardinality, provided it is carefully managed to avoid data leakage during validation.

A particularly relevant aspect of the preprocessing is the creation of nullity indicators. For columns with missing values, binary variables are generated to signal if a record had a null before imputation. In domains like fraud, the strategic absence of data can be a signal in itself, so incorporating this explicit information responds to the idea of not treating nulls solely as a technical problem, but also as a possible indicator of the analyzed phenomenon.

Subsequently, median imputation is applied to numerical variables and standard scaling, leaving the dataset free of nulls and adjusted in terms of scale. On this basis, a Principal Component Analysis (PCA) is performed, preserving 95% of the explained variance, which results in 58 principal components. The final dataset is composed of `TransactionID`, `isFraud` and these 58 components, significantly reducing the dimensionality compared to the nearly 200 original columns.

### 2.4. Analysis and proposed models

At an analytical level, three complementary layers are proposed. The first is basic descriptive analysis: fraud rates, distribution by time slots, behavior by payment methods, absence patterns, and behavioral differences between legitimate and fraudulent operations. This layer allows validating initial hypotheses and detecting obvious relationships.

The second is exploratory multivariate analysis, especially over the PCA space. Representing the transactions in the first components and coloring by `isFraud` can reveal groupings or separations that were not evident in the original variables. This exploration helps to identify regions of the space associated with a higher probability of fraud and to base the subsequent choice of models.

The third layer is predictive analysis. For example, using a logistic regression as an interpretable baseline and tree and ensemble models, such as Random Forest, for their ability to handle non-linear relationships and attributes of diverse nature. But because fraudsters often update their behaviors, any proposed process must be iterative and under constant supervision to foresee behavioral changes of users and fraudsters that can confuse the model.

In this context, evaluation metrics must adapt to the imbalance of the problem, which is why metrics such as F1, recall, and precision become relevant, especially when the minority class —in this case, fraud— is the one that concentrates the main interest. Maximizing recall can reduce false negatives, but a precision that is too low as a consequence would saturate manual review and penalize legitimate customers, so the choice of threshold and model must respond to a compromise between both types of error, represented by the F1-score that seeks a balance between both metrics.

## 3. Design of the NoSQL storage system

### 3.1. SQL vs. NoSQL in the context of the case

SQL databases are highly suitable when the schema is stable, transactional consistency is a priority, and relationships between tables are well defined. However, in Big Data scenarios with semi-structured data, the need for horizontal scaling, and schemas that evolve rapidly and require frequent rethinking to avoid drift in a changing environment, they can be less flexible and more costly to adapt. NoSQL models, on the other hand, offer more efficiency in data processing as a family of solutions adapted to different usage patterns: key-value for very simple and fast access, document for flexible JSON data, columnar for large volumes of series and reads by column groups. Therefore, NoSQL is the appropriate choice for this context.

### 3.2. Choice of a document model with MongoDB

For the fraud case, a document model based on MongoDB is proposed, because the transactions contain a combination of relatively structured attributes with more flexible metadata, which can vary between channels, countries, or external integrations. A document store allows representing each transaction as a much richer JSON document, without imposing a rigid and common schema for all variants, being able to integrate tabular schemas inside as well, so that nothing is lost, but rather gained. MongoDB provides several relevant advantages for this case: natural integration with JSON formats, ability to scale horizontally through sharding, support for agile CRUD operations, and, above all, aggregation pipelines that allow performing complex analyses directly in the database, without having to continuously export data to the outside. This is useful for building operational dashboards, reports, or queries about the behavior of the detection system.

### 3.3. Structure of collections and documents

A minimum structure of collections to organize the data and decisions is proposed as an example:
- `transactions_raw`, where transactions are stored exactly as they arrive, with their origin and ingestion timestamp.
- `transactions_preprocessed`, where the transformed versions are saved, including derived variables, nullity indicators, and principal components.
- `fraud_scores`, with the scoring results: numerical score or binary data directly once verified, predicted label, applied threshold, and model version.
- `alerts_review`, with the operations that are derived to manual review, their status, resolution, and analysts' annotations.
- `reference_metadata`, with catalogs, configurations, and auxiliary rules.

This organization avoids mixing data layers and allows maintaining traceability, something imperative for a good flow between acquisition, processing, storage, and analysis to be coherent. Maintaining both the raw and preprocessed data layers makes it easier to redo transformations, audit results, and adapt models without losing the original history.

### 3.4. Advantages and disadvantages

The document model presents several advantages in this context: schema flexibility, the ability to store nested structures, simple integration with services and microservices, and advanced aggregation support. However, it also has limitations: certain forms of complex transactional consistency may require support from relational engines, and inefficient document design can lead to duplication or costly queries. Therefore, it is suggested that, in larger projects, it could be considered for MongoDB to coexist with relational systems or even with other specialized NoSQL databases, depending on the data and query type.

## 4. Big Data Architecture for fraud detection

### 4.1. Principles and general structure

The proposed Big Data architecture design follows a hybrid logic inspired by Lambda, combining batch processing and streaming processing. This choice is justified because fraud detection requires two simultaneous capabilities: deep analysis of large historical data, to train and fine-tune models because fraudsters often adapt their behavior to mask themselves against these types of models, and rapid response to incoming events, to block or review high-risk operations in real time.

The general flow is divided into six stages: ingestion from multiple sources, distributed raw storage, batch processing for cleaning and modeling, streaming processing for scoring, operational storage in MongoDB, and consumption of results through dashboards and decision systems, as well as a final phase of continuous analysis, review, and feedback.

### 4.2. Ingestion and distributed storage

The ingestion contemplates two types of channels. On one hand, real-time transactional events that are directed to both the streaming layer and the raw storage layer. On the other hand, batch loads of historical data, logs, and external sources, which are periodically incorporated into the dataset (data lake). The distributed storage is implemented on Hadoop/HDFS type technologies in the cloud, taking advantage of the block replication logic, computation distribution, and fault tolerance. Within the data lake, a distinction is made between raw zones (original, raw data), cleansed zones (clean and normalized data), and curated zones (analytical datasets ready for modeling and reporting).

### 4.3. Batch processing with Spark

The batch processing uses Spark to perform massive cleaning, imputation, encoding, scaling, feature engineering, and PCA tasks on large volumes, transferring the logic already applied in the prototype to the distributed environment. Spark allows combining these transformations into reproducible pipelines, and its machine learning libraries facilitate the training of classification models on the prepared data. This layer also assumes the periodic evaluation of models, data drift detection, the generation of new aggregated variables, and the production of updated versions of the fraud models that will be deployed in the streaming layer, thus forming a cyclical process that is more than necessary and even more so in this context where the phenomenon to be detected actively tries to evade detection by the model.

### 4.4. Streaming processing and operational scoring

The streaming layer, supported by Spark Structured Streaming, is responsible for processing transaction events in near real-time. Each event is enriched with fast variables, queries the current scoring model, and generates a risk score and a possible fraud label. High-risk operations are routed to `alerts_review` in MongoDB and to notification systems for manual review or automatic blocking, depending on the business rules the company decides to impose based on historical data. This streaming processing maintains consistency with the distributed processing and fault tolerance paradigm: flows can be resumed, the load can be redistributed among nodes, and the infrastructure can scale horizontally when the volume of events increases.

### 4.5. Consumption and visualization layer

Finally, the architecture includes a consumption layer that offers dashboards and ad hoc analysis for technical and business profiles. For business, indicators such as fraud rate, number of alerts, estimated avoided cost, and distribution by channel and by country are shown. For data science and engineering teams, model metrics (F1, recall, precision), temporal evolution of performance, drift patterns, and principal component analysis are exposed. It should be noted that visualization is not an aesthetic luxury, but a central tool for understanding the phenomenon, communicating findings, and making informed decisions. The proposed architecture respects this principle by placing the visual layer as an integrated part of the flow, not as a marginal addition.